# Chapter 7. Calibration, timing, and session metadata

**Learning objective.** Validate whether a recording-session record is internally consistent with file headers and a declared timing requirement, without confusing documentation with equipment certification.

**Estimated time:** 45–60 minutes. **Prerequisite:** Chapter 7 and the recording-QC notebook. **Exercises:** 7.3 and 7.4.

All bundled inputs are explicitly **synthetic teaching fixtures**. The timing events are generated numerical values. The metadata describe no participant, real site, or physical microphone. Replace them only with institutionally authorized local records.

## Input and output contract

The notebook requires a session YAML record, the WAV manifest, and a timing-event table. It writes `validated_session_metadata.json`, `recording_deviations.tsv`, `timing_check.tsv`, `run_provenance.json`, and `decision_record.json` under `companion/outputs/ch07/02_calibration_and_metadata/`.

The validator checks required fields, WAV headers, file counts, and the residuals of a linear timing relation. A successful run establishes internal consistency only. It does not establish microphone calibration, room suitability, or construct validity.

In [ ]:
from pathlib import Path
import csv
import json
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the repository root or one of its subdirectories.")

SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch07" / "synthetic_recordings"
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "The Chapter 7 fixture is absent. Run companion/scripts/generate_ch07_fixture.py explicitly; "
        "the notebook will not create or substitute data silently."
    )
print(f"Repository: {REPOSITORY_ROOT}")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — no participant observations are present.")

In [ ]:
from phonetic_research_companion.ch07 import validate_session_metadata

METADATA = DATA_ROOT / "recording_session_metadata.yaml"
MANIFEST = DATA_ROOT / "recording_manifest.tsv"
TIMING_EVENTS = DATA_ROOT / "timing_events.tsv"
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch07" / "02_calibration_and_metadata"

validated = validate_session_metadata(METADATA, MANIFEST, TIMING_EVENTS, OUTPUT)
assert validated["metadata"]["data_status"] == "SYNTHETIC_TEACHING_FIXTURE"
assert validated["validation_status"] == "pass"
assert validated["observed_file_count"] == 3
print(json.dumps(validated["timing_fit"], indent=2))

In [ ]:
with (OUTPUT / "recording_deviations.tsv").open(encoding="utf-8", newline="") as source:
    deviations = list(csv.DictReader(source, delimiter="	"))
with (OUTPUT / "timing_check.tsv").open(encoding="utf-8", newline="") as source:
    timing_rows = list(csv.DictReader(source, delimiter="	"))
print(f"Recorded deviations: {len(deviations)}")
print("Timing events:")
for row in timing_rows:
    print(row)

## Learner decision

Decide whether the synthetic session should be accepted, rerun, restricted to named measures, or stopped. Distinguish the deliberately defective teaching files from the timing check, and state why a complete metadata form does not certify the physical accuracy of hardware that was never calibrated.

In [ ]:
LEARNER_DECISION = {
    "data_status": "SYNTHETIC_TEACHING_FIXTURE",
    "decision_status": "undecided",
    "claim_requirements_checked": ["file headers", "file count", "timing residual"],
    "limitations": ["no physical microphone", "no calibrated amplitude", "no participant speech"],
    "rationale": "Learner response required before assessment.",
    "warning": "This clean-run record is not an empirical conclusion and must not be reported as one.",
}
(OUTPUT / "decision_record.json").write_text(json.dumps(LEARNER_DECISION, indent=2), encoding="utf-8")
for required in (
    "validated_session_metadata.json", "recording_deviations.tsv", "timing_check.tsv", "run_provenance.json"
):
    assert (OUTPUT / required).exists(), required
print(json.dumps(LEARNER_DECISION, indent=2))